In [ ]:
import pypsa
from pypsa.common import annuity
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.io as pio
import plotly.offline as py

pd.options.plotting.backend = "plotly"

# Render static PNG images so plots are visible on GitHub
pio.renderers.default = "png"


## Read preprocessed Data 

In [240]:
data_cost : str = "data/costs_2030.csv"


costs = pd.read_csv(data_cost, index_col=[0, 1])

costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
costs.unit = costs.unit.str.replace("/kW", "/MW")

defaults = {
    "FOM": 0,
    "VOM": 0,
    "efficiency": 1,
    "fuel": 0,
    "investment": 0,
    "lifetime": 25,
    "CO2 intensity": 0,
    "discount rate": 0.07,
}

costs = costs.value.unstack().fillna(defaults)
costs.at["OCGT", "fuel"] = costs.at["gas", "fuel"]
costs.at["CCGT", "fuel"] = costs.at["gas", "fuel"]
costs.at["OCGT", "CO2 intensity"] = costs.at["gas", "CO2 intensity"]
costs.at["CCGT", "CO2 intensityuel"] = costs.at["gas", "CO2 intensity"]

costs.rename(index={"onwind": "wind", "solar": "pv"}, inplace=True)


discount_rate = 0.07
lifetime = 20

annuity_factor = annuity(discount_rate, lifetime)


costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]
costs["capital_cost"] = (annuity_factor + costs["FOM"] / 100 ) * costs["investment"]


/var/folders/c2/0pncj6ld4glby0b51j0k8dgc0000gn/T/ipykernel_4471/1558647293.py:32: DeprecatedWarning:

annuity is deprecated as of 1.1.0 and will be removed in 2.0.0. Use pypsa.costs.annuity() instead.



In [241]:
timeseries_path = "data/timeseries_2025_germany.csv"

timeseries = pd.read_csv(timeseries_path, index_col=0, parse_dates=True)

In [242]:
timeseries.resample("4h").first()

,load_Nord,load_West,load_Ost,load_Süd,wind_Nord,wind_West,wind_Ost,wind_Süd,pv_Nord,pv_West,pv_Ost,pv_Süd
time,,,,,,,,,,,,
2025-01-01 00:00:00,8519.080950,15145.032800,9465.645500,14198.468250,0.989543,0.749514,0.850072,0.240686,0.000000,0.000000,0.000000,0.000000
2025-01-01 04:00:00,7966.970550,14163.503200,8852.189500,13278.284250,0.998344,0.822706,0.928523,0.328637,0.000000,0.000000,0.000000,0.000000
2025-01-01 08:00:00,8665.275600,15404.934400,9628.084000,14442.126000,0.951009,0.921379,0.955833,0.419579,0.000034,0.000000,0.008245,0.067405
2025-01-01 12:00:00,9427.708800,16760.371200,10475.232000,15712.848000,0.896804,0.889816,0.972782,0.446723,0.037500,0.275686,0.231281,0.514126
2025-01-01 16:00:00,10135.511100,18018.686400,11261.679000,16892.518500,0.999977,0.989557,0.995142,0.588582,0.000000,0.004053,0.000000,0.019459
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 04:00:00,9230.250801,16409.334757,10255.834223,15383.751335,0.556287,0.113664,0.331164,0.046102,0.000000,0.000000,0.000000,0.000000
2025-12-31 08:00:00,9230.250801,16409.334757,10255.834223,15383.751335,0.737223,0.163983,0.471398,0.078796,0.001038,0.000000,0.041102,0.072906
2025-12-31 12:00:00,9230.250801,16409.334757,10255.834223,15383.751335,0.620009,0.241881,0.721611,0.122856,0.118339,0.200982,0.030758,0.388230


In [243]:
timeseries.columns

Index(['load_Nord', 'load_West', 'load_Ost', 'load_Süd', 'wind_Nord',
       'wind_West', 'wind_Ost', 'wind_Süd', 'pv_Nord', 'pv_West', 'pv_Ost',
       'pv_Süd'],
      dtype='str')

### 4 Zone Model Initialisation

In [244]:
network = pypsa.Network()

In [245]:
carriers = [
    "wind", 
    "pv", # solar
    "OCGT",  # open cycle gas turbine
    "hydrogen storage underground",
    "battery storage",
    "electricity",
]

colors={
    "wind": "dodgerblue",
    "pv": "gold",
    "OCGT": "indianred",
    "hydrogen storage underground": "magenta",
    "battery storage": "yellowgreen",
    "electricity":  "black"}


for carrier in carriers:
    network.add(
        "Carrier",
        name= carrier,
        color = colors[carrier],
        
        co2_emissions = costs.at[carrier, "CO2 intensity"] if carrier in costs.index else 0,
        overwrite=True
        
    )

In [246]:
network.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
name,,,,,
wind,0.000,dodgerblue,,inf,0.0
pv,0.000,gold,,inf,0.0
OCGT,0.198,indianred,,inf,0.0
hydrogen storage underground,0.000,magenta,,inf,0.0
battery storage,0.000,yellowgreen,,inf,0.0
electricity,0.000,black,,inf,0.0


### Adding Buses

In [247]:
regions = ["Nord", "West", "Ost", "Süd"]

for region in regions:
    network.add(
        "Bus",
        name = f"Bus_{region}",
        type = "electricity",
        carrier = 'electricity',
        overwrite=True
    )

In [248]:
network.snapshots = timeseries.index
network.snapshots

DatetimeIndex(['2025-01-01 00:00:00', '2025-01-01 01:00:00',
               '2025-01-01 02:00:00', '2025-01-01 03:00:00',
               '2025-01-01 04:00:00', '2025-01-01 05:00:00',
               '2025-01-01 06:00:00', '2025-01-01 07:00:00',
               '2025-01-01 08:00:00', '2025-01-01 09:00:00',
               ...
               '2025-12-31 14:00:00', '2025-12-31 15:00:00',
               '2025-12-31 16:00:00', '2025-12-31 17:00:00',
               '2025-12-31 18:00:00', '2025-12-31 19:00:00',
               '2025-12-31 20:00:00', '2025-12-31 21:00:00',
               '2025-12-31 22:00:00', '2025-12-31 23:00:00'],
              dtype='datetime64[us]', name='snapshot', length=8760, freq=None)

In [249]:
network.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
name,,,,,,,,,,,,,
Bus_Nord,1.0,electricity,0.0,0.0,electricity,,,1.0,0.0,inf,PQ,,
Bus_West,1.0,electricity,0.0,0.0,electricity,,,1.0,0.0,inf,PQ,,
Bus_Ost,1.0,electricity,0.0,0.0,electricity,,,1.0,0.0,inf,PQ,,
Bus_Süd,1.0,electricity,0.0,0.0,electricity,,,1.0,0.0,inf,PQ,,


In [250]:
network.snapshot_weightings.loc[:,:] = 4 

### Adding Load

In [251]:
regions = ["Nord", "West", "Ost", "Süd"]
for region in regions:
    network.add(
        "Load",
        f"Demand_{region}",
        bus=f"Bus_{region}",
        p_set= timeseries[f"load_{region}"],
        carrier = "electricity",
        overwrite=True
    )

In [252]:
network.loads_t

{'p_set': name                 Demand_Nord   Demand_West    Demand_Ost    Demand_Süd
 snapshot                                                                  
 2025-01-01 00:00:00  8519.080950  15145.032800   9465.645500  14198.468250
 2025-01-01 01:00:00  8323.858350  14797.970400   9248.731500  13873.097250
 2025-01-01 02:00:00  8109.321750  14416.572000   9010.357500  13515.536250
 2025-01-01 03:00:00  7928.587350  14095.266400   8809.541500  13214.312250
 2025-01-01 04:00:00  7966.970550  14163.503200   8852.189500  13278.284250
 ...                          ...           ...           ...           ...
 2025-12-31 19:00:00  9230.250801  16409.334757  10255.834223  15383.751335
 2025-12-31 20:00:00  9230.250801  16409.334757  10255.834223  15383.751335
 2025-12-31 21:00:00  9230.250801  16409.334757  10255.834223  15383.751335
 2025-12-31 22:00:00  9230.250801  16409.334757  10255.834223  15383.751335
 2025-12-31 23:00:00  9230.250801  16409.334757  10255.834223  15383.751335
 
 

In [253]:
network.loads_t.p_set.plot(labels=dict(value = "Load MW"))
# plt.ylabel("Load [MW]")

### Adding Generators

In [254]:
regions = ["Nord", "West", "Ost", "Süd"]
generators = ["wind", "pv", "OCGT"]

for region in regions:
    for generator in generators:
        network.add(
            "Generator",
            name=f"{generator}_{region}",
            bus=f"Bus_{region}",
            carrier=generator,
            p_max_pu=timeseries[f"{generator}_{region}"] if generator != "OCGT" else 1.0,
            marginal_cost=costs.at[generator, "marginal_cost"],
            capital_cost=costs.at[generator, "capital_cost"],
            p_nom_extendable=True,
            overwrite=True
        )

In [255]:
network.generators_t.p_max_pu[:5]

name,wind_Nord,pv_Nord,wind_West,pv_West,wind_Ost,pv_Ost,wind_Süd,pv_Süd
snapshot,,,,,,,,
2025-01-01 00:00:00,0.989543,0.0,0.749514,0.0,0.850072,0.0,0.240686,0.0
2025-01-01 01:00:00,0.992777,0.0,0.779992,0.0,0.861112,0.0,0.257526,0.0
2025-01-01 02:00:00,0.995757,0.0,0.798748,0.0,0.887177,0.0,0.277494,0.0
2025-01-01 03:00:00,0.995413,0.0,0.819556,0.0,0.911713,0.0,0.307796,0.0
2025-01-01 04:00:00,0.998344,0.0,0.822706,0.0,0.928523,0.0,0.328637,0.0


In [256]:
network.generators_t.p_max_pu.loc["2025-06"].plot(labels=dict(value="Capacity Factor [p.u.]"))

### Adding Transmission Lines

In [257]:
lines = {
    "Nord_to_West": ("Bus_Nord", "Bus_West", 300),
    "Nord_to_Ost":  ("Bus_Nord", "Bus_Ost",  400),
    "West_to_Süd":  ("Bus_West", "Bus_Süd",  500),
    "Ost_to_Süd":   ("Bus_Ost",  "Bus_Süd",  400),
}

for name, (bus0, bus1, length) in lines.items():
    network.add(
        "Line",
        name=name,
        bus0=bus0,
        bus1=bus1,
        type="Al/St 240/40 4-bundle 380.0",
        length=length,
        s_nom=2000,
        s_nom_extendable=True,
        overwrite=True
    )


In [258]:
network.calculate_dependent_values()
network.lines

,bus0,bus1,type,x,r,g,b,s_nom,s_nom_mod,s_nom_extendable,...,v_ang_max,sub_network,x_pu,r_pu,g_pu,b_pu,x_pu_eff,r_pu_eff,s_nom_opt,v_nom
name,,,,,,,,,,,,,,,,,,,,,
Nord_to_West,Bus_Nord,Bus_West,Al/St 240/40 4-bundle 380.0,73.8,9.0,0.0,0.001301,2000.0,0.0,True,...,inf,,73.8,9.0,0.0,0.001301,73.8,9.0,0.0,1.0
Nord_to_Ost,Bus_Nord,Bus_Ost,Al/St 240/40 4-bundle 380.0,98.4,12.0,0.0,0.001734,2000.0,0.0,True,...,inf,,98.4,12.0,0.0,0.001734,98.4,12.0,0.0,1.0
West_to_Süd,Bus_West,Bus_Süd,Al/St 240/40 4-bundle 380.0,123.0,15.0,0.0,0.002168,2000.0,0.0,True,...,inf,,123.0,15.0,0.0,0.002168,123.0,15.0,0.0,1.0
Ost_to_Süd,Bus_Ost,Bus_Süd,Al/St 240/40 4-bundle 380.0,98.4,12.0,0.0,0.001734,2000.0,0.0,True,...,inf,,98.4,12.0,0.0,0.001734,98.4,12.0,0.0,1.0


### Adding StorageUnits (Batterie, Hydro Storage)

In [259]:
regions = ["Nord", "West", "Ost", "Süd"]
capital_cost_batterie = costs.at["battery inverter", "capital_cost"] 
+ 4 * costs.at["battery storage", "capital_cost"]

capital_costs_hydro_storage = (
costs.at["electrolysis", "capital_cost"]
+ costs.at["fuel cell", "capital_cost"]
+ 168 * costs.at["hydrogen storage underground", "capital_cost"]
)

for region in regions:
    network.add(
        "StorageUnit",
        name = f"BatteryStorage_{region}",
        carrier= "battery storage",
        bus=f"Bus_{region}",
        max_hours = 4,
        capital_cost = capital_cost_batterie,   
        efficiency_store=costs.at["battery inverter", "efficiency"],
        efficiency_dispatch=costs.at["battery inverter", "efficiency"],
        cyclic_state_of_charge=True, # the state of charge at the beginning of the optimisation period must equal the final state of charge.
        p_nom_extendable=True,
        overwrite=True
    )
    network.add(
        "StorageUnit",
        name = f"hydrogenStorage_{region}",
        bus=f"Bus_{region}",
        carrier="hydrogen storage underground",
        max_hours=168,
        capital_cost=capital_costs_hydro_storage,
        efficiency_store=costs.at["electrolysis", "efficiency"],
        efficiency_dispatch=costs.at["fuel cell", "efficiency"],
        p_nom_extendable=True,
        cyclic_state_of_charge=True,
        overwrite=True
    )


In [260]:
network.storage_units

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,state_of_charge_initial_per_period,state_of_charge_set,cyclic_state_of_charge,cyclic_state_of_charge_per_period,max_hours,efficiency_store,efficiency_dispatch,standing_loss,inflow,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
BatteryStorage_Nord,Bus_Nord,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.9600,0.96,0.0,0.0,0.0
hydrogenStorage_Nord,Bus_Nord,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,168.0,0.6217,0.50,0.0,0.0,0.0
BatteryStorage_West,Bus_West,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.9600,0.96,0.0,0.0,0.0
hydrogenStorage_West,Bus_West,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,168.0,0.6217,0.50,0.0,0.0,0.0
BatteryStorage_Ost,Bus_Ost,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.9600,0.96,0.0,0.0,0.0
hydrogenStorage_Ost,Bus_Ost,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,168.0,0.6217,0.50,0.0,0.0,0.0
BatteryStorage_Süd,Bus_Süd,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,4.0,0.9600,0.96,0.0,0.0,0.0
hydrogenStorage_Süd,Bus_Süd,PQ,,0.0,0.0,True,0.0,inf,NaN,-1.0,...,False,NaN,True,False,168.0,0.6217,0.50,0.0,0.0,0.0


### Adding Emission Limits

In [261]:
# Moderate reduction scenario constant = 80e6    # 80 million tonnes

network.add(
    "GlobalConstraint",
    name="co2_limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant= 80e6
    # constant=0,
)

### Consistency-Check

In [262]:

network.consistency_check()

### Model Run

In [263]:
network.optimize(solver_name="highs")

/var/folders/c2/0pncj6ld4glby0b51j0k8dgc0000gn/T/ipykernel_4471/3589273427.py:1: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

and branches. Passive flows are not allowed for non-electric networks!
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 8/8 [00:00<00:00, 83.59it/s]
INFO:linopy.io: Writing time: 1.24s


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-1h14k83x has 814705 rows; 350424 cols; 1734986 nonzeros
Coefficient ranges:
  Matrix  [1e-07, 2e+06]
  Cost    [5e-02, 5e+05]
  Bound   [0e+00, 0e+00]
  RHS     [6e+03, 8e+07]
Presolving model
412203 rows, 333382 cols, 1245366 nonzeros  1s
Dependent equations search running on 84488 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.02s (limit = 1000.00s)
382811 rows, 303990 cols, 1224998 nonzeros  3s
Presolve reductions: rows 382811(-431894); columns 303990(-46434); nonzeros 1224998(-509988) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 14408(2.28943e+10); Du: 0(2.76324e-09) 4.5s
       1464     1.5057861472e+09 Pr: 67391(9.28174e+12); Du: 0(5.26327e-07) 10.7s
       7061     7.2953392856e+09 Pr: 41714(8.71895e+10); Du: 0(4.09142e-06) 1

INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 350424 primals, 814705 duals
Objective: 5.78e+10
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-ext-p-lower, Generator-ext-p-upper, Line-ext-s-lower, Line-ext-s-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance were not assigned to the network.


('ok', 'optimal')

In [264]:
print(network.objective / 1e9)

57.765128887026606


In [265]:
network.statistics.optimal_capacity().div(1e3)  # GW
# or more detailed:
network.generators.p_nom_opt.div(1e3)           # GW, generators only

name
wind_Nord    125.514253
pv_Nord       21.241012
OCGT_Nord      0.311721
wind_West     -0.000000
pv_West       -0.000000
OCGT_West     11.310230
wind_Ost      -0.000000
pv_Ost        -0.000000
OCGT_Ost       9.360813
wind_Süd      -0.000000
pv_Süd       161.191678
OCGT_Süd      14.693515
Name: p_nom_opt, dtype: float64

In [266]:
network.statistics.energy_balance().sort_values().div(1e6)  # TWh

component    carrier          bus_carrier
Load         electricity      electricity   -1882.306915
StorageUnit  battery storage  electricity     -21.506243
Generator    OCGT             electricity     180.798450
             pv               electricity     860.457854
             wind             electricity     862.556855
dtype: float64

In [267]:
tsc = pd.concat({
    "capex": network.statistics.capex(),
    "opex":  network.statistics.opex(),
}, axis=1).div(1e9).round(2)  # bn€/a
print(tsc)

                             capex   opex
component   carrier                      
Generator   OCGT              2.33  13.62
            pv               14.19   0.01
            wind             18.50   1.56
StorageUnit battery storage   7.56    NaN


In [268]:
emissions = (
    network.generators_t.p
    / network.generators.efficiency
    * network.generators.carrier.map(network.carriers.co2_emissions)
)
print(emissions.sum().sum() / 1e6)  # total Mt CO2/a

8.94952326713398


In [269]:
network.statistics.energy_balance.iplot()